# ViT, Densenet, Resnet

Imports

In [ ]:
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score
import os

Device selection

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA GPU")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple MPS GPU")
else:
    device = torch.device("cpu")
    print("Using CPU")

Data

In [ ]:
#drive.mount('/content/drive')

In [ ]:
DATA_DIR = "/content/drive/MyDrive/aml-2025-feathers-in-focus"

TRAIN_CSV = os.path.join(DATA_DIR, "train_images.csv")
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "train_images")
TEST_IMG_DIR  = os.path.join(DATA_DIR, "test_images")

print("TRAIN_IMG_DIR:", TRAIN_IMG_DIR)
print("CSV exists?", os.path.exists(TRAIN_CSV))
print("Example image exists?", len(os.listdir(TRAIN_IMG_DIR)) > 0)

In [ ]:
full_df = pd.read_csv(TRAIN_CSV)
print(full_df.head())

num_classes = full_df["label"].nunique()
print("Classes:", num_classes)

In [ ]:
class BirdDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

        self.classes = sorted(self.df['label'].unique())
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_rel = str(row["image_path"])
        filename = os.path.basename(img_rel)
        full_path = os.path.join(self.img_dir, filename)

        img = Image.open(full_path).convert("RGB")
        if self.transform:
            img = self.transform(img)

        label = self.class_to_idx[row["label"]]
        return img, label


Train/test split

In [ ]:
train_df, val_df = train_test_split(full_df, test_size=0.2, random_state=42, stratify=full_df['label'])

In [ ]:
BATCH_SIZE = 64
#big bc it was ran on cuda

Data augmentation

In [ ]:
train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(50),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


train_dataset = BirdDataset(train_df, TRAIN_IMG_DIR, transform=train_tfms)
val_dataset   = BirdDataset(val_df,   TRAIN_IMG_DIR, transform=test_tfms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory = True, persistent_workers = True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory = True, persistent_workers = True)

In [ ]:
classes = train_df['label'].unique()
classes

## Model selection

### Basic ViT (no pretraining)

In [ ]:
import torch
import torch.nn as nn
import timm

class BirdViT(nn.Module):
    def __init__(self, num_classes: int, pretrained: bool = False):
        super().__init__()

        self.backbone = timm.create_model(
            "deit_tiny_patch16_224",
            pretrained=pretrained,
            num_classes=num_classes
        )

    def forward(self, x):
        return self.backbone(x)

In [ ]:
num_classes = 200  # <-- set to your number of bird species
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BirdViT(num_classes=num_classes, pretrained=True).to(device)

### DenseNet

In [ ]:
from torchvision.models import densenet121

model = densenet121(weights=None)  # no pretraining
model.classifier = nn.Linear(model.classifier.in_features, num_classes)

### ResNet

In [ ]:
import torch.nn as nn
from torchvision.models import resnet18


class BirdResNet18(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        # weights=None → random init → NOT transfer learning
        self.backbone = resnet18(weights=None)
        # replace the final FC layer
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.backbone(x)

In [ ]:
model = BirdResNet18(num_classes=num_classes).to(device)

## Model Training

In [ ]:
NUM_EPOCHS = 60

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = model.to(device)

In [ ]:
print("CUDA available:", torch.cuda.is_available())
print("Device variable:", device)
print("Model is on:", next(model.parameters()).device)

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
from torch.optim.lr_scheduler import ReduceLROnPlateau

scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=5, min_lr=1e-4)

For ResNet

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01,
    momentum=0.9,
    weight_decay=1e-4
)

from torch.optim.lr_scheduler import CosineAnnealingLR

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

In [ ]:
best_val = 0

from tqdm.notebook import tqdm
for epoch in range(NUM_EPOCHS):
    # ---- TRAIN ----
    model.train()
    train_loss = 0.0
    train_correct = 0
    total = 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} Training"):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        train_correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_loss /= total
    train_acc = 100 * train_correct / total

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} Validating"):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = 100 * val_correct / val_total

    if val_acc > best_val:
        best_val = val_acc
        torch.save(model.state_dict(), "best_model_resnet.pth")
        print("Saved new best model!")

    scheduler.step(val_acc)
    current_lr = optimizer.param_groups[0]["lr"]

    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val  Loss: {val_loss:.4f} | Val  Acc: {val_acc:.2f}%")
    print(f"LR: {current_lr:.6f}")

## Testing

In [ ]:
#model.load_state_dict(torch.load("best_model_vit.pth", map_location=device))
#model.eval()

In [ ]:
EST_IMG_DIR   = os.path.join(DATA_DIR, "test_images")
TEST_PATH_CSV  = os.path.join(DATA_DIR, "test_images_path.csv")
SAMPLE_SUB_CSV = os.path.join(DATA_DIR, "test_images_sample.csv")

print("TEST_IMG_DIR:", TEST_IMG_DIR)
print("TEST_PATH_CSV exists?", os.path.exists(TEST_PATH_CSV))
print("SAMPLE_SUB_CSV exists?", os.path.exists(SAMPLE_SUB_CSV))

IMG_SIZE = (224, 224)

test_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

In [ ]:
class BirdTestDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # assumes column is called "image_path"
        img_rel = str(row["image_path"])
        filename = os.path.basename(img_rel)
        full_path = os.path.join(self.img_dir, filename)

        img = Image.open(full_path).convert("RGB")
        if self.transform:
            img = self.transform(img)

        return img

In [ ]:
test_paths_df = pd.read_csv(TEST_PATH_CSV)
print("test_images_path.csv head:")
print(test_paths_df.head())

test_dataset = BirdTestDataset(test_paths_df, TEST_IMG_DIR, transform=test_transform)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True,
    persistent_workers=True)

print("Test samples:", len(test_dataset))

In [ ]:
from tqdm.notebook import tqdm
import numpy as np

model.eval()
test_preds = []

with torch.no_grad():
    for images in tqdm(test_loader, desc="Predicting on test"):
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(1) 
        preds = preds + 1          
        test_preds.extend(preds.cpu().numpy())

test_preds = np.array(test_preds)
print("Num predictions:", len(test_preds))

Test time augmentation

In [ ]:
import torch
from tqdm.notebook import tqdm
import numpy as np

tta_transforms = [
    test_tfms,
    transforms.Compose([
        transforms.Resize(IMG_SIZE),
        transforms.functional.hflip,
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ]),
]

model.eval()
all_preds = []

with torch.no_grad():
    for images in tqdm(test_loader, desc="TTA predicting on test"):
        images = images.to(device, non_blocking=True)

        logits_sum = model(images)

        flipped = torch.flip(images, dims=[3])
        logits_sum += model(flipped)

        logits_avg = logits_sum / 2.0

        preds = logits_avg.argmax(dim=1) + 1
        all_preds.extend(preds.cpu().numpy())

test_preds = np.array(all_preds)
print("Num predictions:", len(test_preds))

In [ ]:
sample_df = pd.read_csv(SAMPLE_SUB_CSV)
print("Sample submission head:")
print(sample_df.head())
print("sample_df len:", len(sample_df), "  preds len:", len(test_preds))

if len(sample_df) != len(test_preds):
    print("⚠️ Length mismatch, check test_paths_df vs sample_df.")
else:
    sample_df["label"] = test_preds
    submission_path = "submission.csv"
    sample_df.to_csv(submission_path, index=False)
    print("Saved submission to:", submission_path)
    display(sample_df.head())